# Quick check for `data_preparation`
Small notebook to test SARIMAX-ready preparation and train/test split.


In [4]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'forecast_hourly_base').exists() and (ROOT.parent / 'forecast_hourly_base').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from forecast_hourly_base.data_preparation import (
    prepare_train_for_sarimax,
    prepare_inference_for_sarimax,
    split_panel_train_test,
)

DATA_DIRS = [ROOT / 'src' / 'data', ROOT / 'data']
DATA_DIR = next((p for p in DATA_DIRS if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f'No data directory found in: {DATA_DIRS}')

print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)

attraction_name = "Bumper Cars"

ROOT: c:\Users\joshu\Academic Projecr\ArturCode\Hackathon-Eleven-Strategy\src
DATA_DIR: c:\Users\joshu\Academic Projecr\ArturCode\Hackathon-Eleven-Strategy\src\data


In [13]:
pack = prepare_train_for_sarimax(data_dir=DATA_DIR, attraction_name = None, train_ratio=0.8)
full_df = pack['full_df']
train_df = pack['train_df']
test_df = pack['test_df']
exog_cols = pack['exog_cols']

print('full shape:', full_df.shape)
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('split timestamp:', pack['split_timestamp'])
print('num exog cols:', len(exog_cols))
print('num dummy cols:', len(pack['dummy_cols_created']))
full_df.head()


full shape: (294136, 30)
train shape: (238769, 30)
test shape: (55367, 30)
split timestamp: 2021-12-12 20:00:00
num exog cols: 27
num dummy cols: 17


,date_hour,ENTITY_DESCRIPTION_SHORT,wait_time_avg,attendance,temp,pressure,humidity,wind_speed,clouds_all,rain_1h,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,2018-07-21 09:00:00,Bumper Cars,5.0,59066.0,20.59,1014,68,3.43,62,0.0,...,0,0,0,0,1,0,0,0,0,0
1,2018-07-21 10:00:00,Bumper Cars,5.0,59066.0,22.28,1014,62,2.41,66,0.0,...,0,0,0,0,1,0,0,0,0,0
2,2018-07-21 11:00:00,Bumper Cars,12.5,59066.0,23.20,1014,58,2.39,59,0.0,...,0,0,0,0,1,0,0,0,0,0
3,2018-07-21 12:00:00,Bumper Cars,17.5,59066.0,24.70,1014,56,2.10,34,0.0,...,0,0,0,0,1,0,0,0,0,0
4,2018-07-21 13:00:00,Bumper Cars,12.5,59066.0,24.71,1014,54,2.27,51,0.0,...,0,0,0,0,1,0,0,0,0,0


## Testing

In [6]:
test_df.head()

,date_hour,ENTITY_DESCRIPTION_SHORT,wait_time_avg,attendance,temp,pressure,humidity,wind_speed,clouds_all,rain_1h,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,2021-12-28 21:00:00,Bumper Cars,5.00,49787.0,10.48,1007,82,6.40,91,0.00,...,0,0,0,0,0,0,0,0,0,1
1,2021-12-29 09:00:00,Bumper Cars,5.00,45147.0,9.48,1009,99,4.06,100,1.01,...,0,0,0,0,0,0,0,0,0,1
2,2021-12-29 10:00:00,Bumper Cars,11.25,45147.0,10.61,1009,99,3.83,100,0.26,...,0,0,0,0,0,0,0,0,0,1
3,2021-12-29 11:00:00,Bumper Cars,27.50,45147.0,11.73,1009,99,6.23,100,0.00,...,0,0,0,0,0,0,0,0,0,1
4,2021-12-29 12:00:00,Bumper Cars,37.50,45147.0,13.19,1010,99,6.38,100,0.15,...,0,0,0,0,0,0,0,0,0,1


In [7]:
print('train max time:', train_df['date_hour'].max())
print('test  min time:', test_df['date_hour'].min())
print('time split OK:', train_df['date_hour'].max() <= test_df['date_hour'].min())

train max time: 2021-12-28 20:00:00
test  min time: 2021-12-28 21:00:00
time split OK: True


## Inference

In [8]:
weather_forecast_df = (
    pd.read_csv(
        DATA_DIR / 'weather_data.csv',
        usecols=['dt_iso', 'temp', 'pressure', 'humidity', 'wind_speed', 'clouds_all', 'rain_1h', 'visibility'],
    )
    .tail(24 * 7)
    .copy()
)

attendance_forecast_df = (
    pd.read_csv(DATA_DIR / 'attendance.csv')
    .query("FACILITY_NAME == 'PortAventura World'")
    .assign(date=lambda d: pd.to_datetime(d['USAGE_DATE'], errors='coerce').dt.floor('D'))
    [['date', 'attendance']]
    .dropna()
    .drop_duplicates('date')
    .tail(7)
)

previous_week_real_df = (
    full_df[full_df['ENTITY_DESCRIPTION_SHORT'].astype(str) == attraction_name]
    [['date_hour', 'guests_sum', 'availability', 'utilization']]
    .tail(24 * 7)
    .copy()
)

print('attraction:', attraction_name)
print('weather rows:', len(weather_forecast_df))
print('attendance days:', len(attendance_forecast_df))
print('previous-week rows:', len(previous_week_real_df))


attraction: Bumper Cars
weather rows: 168
attendance days: 7
previous-week rows: 168


In [9]:
attendance_forecast_df = (
    pd.read_csv(DATA_DIR / 'attendance.csv')
    .query("FACILITY_NAME == 'PortAventura World'")
    .assign(date=lambda d: pd.to_datetime(d['USAGE_DATE'], errors='coerce').dt.floor('D'))
    [['date', 'attendance']]
    .dropna()
    .drop_duplicates('date')
)

In [10]:
inference_df = prepare_inference_for_sarimax(
    weather_forecast_df=weather_forecast_df,
    attendance_forecast_df=attendance_forecast_df,
    attraction_name=attraction_name,
    previous_week_real_df=previous_week_real_df,
    train_feature_cols=exog_cols,
)

missing_exog = [c for c in exog_cols if c not in inference_df.columns]
print('inference shape:', inference_df.shape)
print('missing exog cols:', len(missing_exog))
inference_df.head()


inference shape: (168, 30)
missing exog cols: 0


,date_hour,ENTITY_DESCRIPTION_SHORT,wait_time_avg,attendance,temp,pressure,humidity,wind_speed,clouds_all,rain_1h,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,2022-08-17 00:00:00,Bumper Cars,NaN,0.0,17.89,1010,97,2.53,100,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2022-08-17 01:00:00,Bumper Cars,NaN,0.0,17.99,1010,97,2.53,100,0.0,...,0,0,0,0,0,0,0,0,0,0
2,2022-08-17 02:00:00,Bumper Cars,NaN,0.0,17.65,1009,97,1.52,100,0.0,...,0,0,0,0,0,0,0,0,0,0
3,2022-08-17 03:00:00,Bumper Cars,NaN,0.0,17.36,1009,97,1.52,100,0.0,...,0,0,0,0,0,0,0,0,0,0
4,2022-08-17 04:00:00,Bumper Cars,NaN,0.0,17.10,1009,98,1.52,100,0.0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
attendance_forecast_df

,date,attendance
0,2018-06-01,46804
2,2018-06-02,57940
4,2018-06-03,44365
6,2018-06-04,37617
8,2018-06-05,32438
...,...,...
2357,2022-07-22,49586
2359,2022-07-23,51748
2361,2022-07-24,45261
2363,2022-07-25,53764


In [12]:
train2, test2 = split_panel_train_test(full_df, train_ratio=0.8, time_col='date_hour')
print('split_panel_train_test shapes:', train2.shape, test2.shape)
print('same split as wrapper:', train2.shape == train_df.shape and test2.shape == test_df.shape)


split_panel_train_test shapes: (10775, 30) (2694, 30)
same split as wrapper: True
